# Project FORESIGHT — Phase 3A: Leakage-Safe Feature Engineering

**Objective**: Construct, validate, and persist a production-grade, strictly leakage-safe feature engineering pipeline for weekly demand forecasting.

**Strict Boundaries**:
- NO machine learning model training or hyperparameter tuning.
- NO mutation of raw CSV files or `data/processed/analysis_ready.parquet`.
- Direct multi-horizon target formulation: `target_h1` through `target_h8`.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add project root to sys.path
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CFG, PATHS
from src.feature_engineering import (
    build_weekly_panel,
    build_multi_horizon_targets,
    build_model_features,
    generate_feature_dictionary,
    generate_feature_statistics,
    validate_feature_leakage,
    FeatureEngineer,
)

print(f'Feature Engineering Framework initialized. Random seed: {CFG.random_seed}')


## 1. Load Analysis-Ready Data & Construct Features

We construct the complete feature table and filter to valid training rows (52 weeks historical warm-up for `lag_52`, plus complete 8-week forward target availability).


In [2]:
df_daily = pd.read_parquet(PATHS.processed_dir / 'analysis_ready.parquet')
df_full, df_valid = build_model_features(df_daily)
print(f'Full Panel Shape: {df_full.shape} (50 SKUs x 106 weeks)')
print(f'Valid Training Shape: {df_valid.shape} (50 SKUs x 47 valid origins)')
print(f'Origins Date Range: {df_valid["forecast_origin_date"].min().strftime("%Y-%m-%d")} to {df_valid["forecast_origin_date"].max().strftime("%Y-%m-%d")}')


## 2. Feature Governance & Leakage Status

Every feature is governed in a formal dictionary. Review the feature distribution by leakage status and group.


In [3]:
dict_df = generate_feature_dictionary()
print('Feature Governance by Leakage Status:')
print(dict_df['leakage_status'].value_counts())
print('\nFeature Count by Feature Group:')
print(dict_df['feature_group'].value_counts())


## 3. Missingness & Target Verification

Verify that in the valid training panel, all 50 features have zero missing values and all 8 forward targets are populated.


In [4]:
null_counts = df_valid.isna().sum()
assert (null_counts == 0).all(), 'Found unexpected nulls!'
print('Feature Missingness: 0.00% across all columns. [OK]')
target_cols = [f'target_h{h}' for h in range(1, 9)]
print('Target summary:')
print(df_valid[target_cols].describe().round(2))


## 4. Visual Diagnostics


In [5]:
plots_dir = PROJECT_ROOT / 'artifacts' / 'features' / 'plots'
plots = sorted(list(plots_dir.glob('*.png')))
print(f'Found {len(plots)} diagnostic plots:')
for p in plots:
    print(f'  - {p.name}')


## 5. Automated Leakage Validation


In [6]:
is_valid = validate_feature_leakage(df_valid)
print(f'Automated Leakage Invariant Check: {"PASSED [OK]" if is_valid else "FAILED"}')
